# 26. Random Forest and matched three-model trajectory analysis

Next step after reviewing the prior team report. This notebook retains the accepted protocol:60-calendar-day rolling training intervals,14 within-window warmup days,7-day refits, daily direct1/2/3-day forecasts. Same585 ZIPs and243 development origins in2020. All34 candidate features, including bothPressure measures, retained. ZIP identity uses585 one-hot columns, not an ordered numeric postal code; expanded input width619. Dynamic Panel uses regionFE;XGBoost uses categorical identity. These representations differ, but no new epidemiological information is supplied.

Random Forest parameters are fixed before viewing its results:200 trees,depth8,min_leaf20,max_features1.0,bootstrap0.8,n_jobs4,seed20261027. This is a declared starting specification, not a tuned bestRandomForest. No OOB-based model selection, no feature selection, no Pressure ablation and no seasonal naive.2021–2023 evaluation remains off. Training targets must mature by fit date. Existing retrospective cohort and prior exploration prevent an untouched-test claim.

Review of prior report: its30-dayrolling buffer is recursive prediction input, not documented weekly refitting. Its GMM studies use three-month training blocks to predict a subsequent month; we borrow forward chronological evaluation and stage-specific reporting, not its exact lag/interval claims. See research_notes/prior_report_review_20261027/REPORT_REVIEW_AND_NEXT_STEPS.md.

Requiresnumpy,pandas,scipy,scikit-learn,joblib,matplotlib. Experiment run usedscikit-learn1.9.0. Documentation:https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
GROUPS={
 'core_history':['cases_today','cases_lag_1','cases_lag_7','cases_lag_14'],
 'rolling_level':['cases_mean_7d','cases_mean_14d'],
 'volatility':['case_std_7d','case_std_14d'],
 'dynamics':['case_acceleration','case_jerk'],
 'local_rates':['case_rate_today_per_100k','case_rate_7d_per_100k','case_rate_14d_per_100k'],
 'sparse_history':['days_since_last_observed_case_censored','no_previous_observed_case'],
 'pressure':['nj_covid_pressure_7d','nj_covid_pressure_14d'],
 'month':[f'target_month_{k}' for k in range(2,13)],
 'weekday':[f'target_weekday_{k}' for k in range(2,8)]}

def run_random_forest60(source, cohort_file, output, stage='development', evaluation_plan_frozen=False):
    if stage not in ['development','evaluation']: raise ValueError('Unknown stage')
    if stage=='evaluation' and not evaluation_plan_frozen:
        raise ValueError('Freeze model/features/hyperparameters before running subsequent evaluation.')
    from sklearn.ensemble import RandomForestRegressor
    from scipy import sparse
    import sklearn,platform,joblib,time
    params=dict(n_estimators=200,max_depth=8,min_samples_leaf=20,
                max_features=1.0,bootstrap=True,max_samples=0.8,
                criterion='squared_error',random_state=20261027,n_jobs=4)
    source,output=Path(source),Path(output)
    output.mkdir(parents=True,exist_ok=True)
    with source.open('rb') as f:
        if f.read(7)==b'version': raise ValueError('Input is a Git LFS pointer. Download the real gzip data first.')
    models=output/'models';models.mkdir(exist_ok=True)
    importance=[]
    ids=pd.read_csv(cohort_file,dtype={'zip':str})['zip'].to_numpy(dtype='U5')
    numeric=sum([v for k,v in GROUPS.items() if k not in ['month','weekday','sparse_history']],[])
    cols=['zip','date','pressure_definition']+numeric
    frame=pd.read_csv(source,usecols=cols,dtype={'zip':str},parse_dates=['date'])
    dates=pd.date_range('2020-03-01','2023-12-31')
    frame=frame.loc[frame.zip.isin(ids)&frame.date.isin(dates)]
    assert len(ids)==585 and not frame.duplicated(['zip','date']).any()
    assert frame.pressure_definition.eq('valid_population_nj_origins_renormalized_v2').all()
    arrays={c:frame.pivot(index='zip',columns='date',values=c).reindex(index=ids,columns=dates).to_numpy(float) for c in numeric}
    counts=arrays['cases_today'];features=sum(GROUPS.values(),[])
    assert np.isfinite(counts).all() and (counts>=0).all()
    # End-of-day origins: initial fit includes mature targets through April29.
    # A 60-day interval includes14 warmup days, leaving45/44/43 pairs per ZIP.
    first=int(dates.get_loc('2020-04-29'))
    origin_start='2020-04-30' if stage=='development' else '2021-01-01'
    origin_end='2020-12-28' if stage=='development' else '2023-12-28'
    origins=np.flatnonzero((dates>=origin_start)&(dates<=origin_end))
    actual=np.stack([counts[:,origins+h] for h in (1,2,3)],axis=2)
    pred=np.full_like(actual,np.nan);logs=[];coefficients=[];omitted=[]
    # Anchor weekly schedule to April29, keep same schedule across stage boundaries.
    fit_indices=np.unique(first+7*((origins-first)//7))
    for fit in fit_indices:
        lo=fit-59
        batch=np.flatnonzero((origins>=fit)&(origins<fit+7))
        assert lo>=0 and len(batch)>0
        last=np.full(len(ids),-1,dtype=int)
        gap=np.empty((len(ids),len(dates)));never=np.empty_like(gap)
        gap[:]=np.nan;never[:]=np.nan
        for t in range(lo,max(fit,int(origins[batch[-1]]))+1):
            last=np.where(counts[:,t]>0,t,last)
            never[:,t]=(last<0);gap[:,t]=np.where(last>=0,t-last,t-lo+1)
        arrays['no_previous_observed_case']=never
        arrays['days_since_last_observed_case_censored']=gap
        for hh,h in enumerate((1,2,3)):
            train=np.arange(lo+14,fit-h+1)
            target=dates+pd.Timedelta(days=h)
            for k in range(2,13): arrays[f'target_month_{k}']=np.broadcast_to((target.month==k).astype(float),counts.shape)
            for k in range(2,8): arrays[f'target_weekday_{k}']=np.broadcast_to((target.dayofweek==k-1).astype(float),counts.shape)
            xt=np.stack([arrays[f][:,train] for f in features],axis=2)
            xp=np.stack([arrays[f][:,origins[batch]] for f in features],axis=2)
            yt=counts[:,train+h]
            assert np.isfinite(xt).all() and np.isfinite(xp).all()
            assert train[0]-14>=lo and train[-1]+h<=fit and np.all(origins[batch]+h>fit)
            # Numeric epidemiologic/calendar columns plus one-hot ZIP identity.
            # Never treat ZIP numbers as an ordered numeric covariate.
            def design(values,fmt):
                rows=len(ids)*values.shape[1]
                identity=sparse.csr_matrix((np.ones(rows,dtype=np.float32),
                          (np.arange(rows),np.repeat(np.arange(len(ids)),values.shape[1]))),
                          shape=(rows,len(ids)))
                return sparse.hstack([sparse.csr_matrix(values.reshape(rows,len(features)).astype(np.float32)),identity],format=fmt)
            train_x,predict_x=design(xt,'csc'),design(xp,'csr')
            model=RandomForestRegressor(**params)
            model.fit(train_x,yt.ravel()) # no validation-based tuning or OOB selection
            pred[:,batch,hh]=model.predict(predict_x).reshape(len(ids),len(batch))
            joblib.dump(model,models/f'rf_{dates[fit].date()}_h{h}.joblib',compress=3)
            importance_names=features+['region_'+zz for zz in ids]
            for feature,value in zip(importance_names,model.feature_importances_):
                if value>0: importance.append(dict(fit_date=str(dates[fit].date()),horizon=h,feature=feature,training_impurity_importance=float(value)))
            logs.append(dict(fit_date=str(dates[fit].date()),window_start=str(dates[lo].date()),window_end=str(dates[fit].date()),window_calendar_days=60,horizon=h,training_origin_start=str(dates[train[0]].date()),training_origin_end=str(dates[train[-1]].date()),latest_training_target=str(dates[train[-1]+h].date()),usable_training_origins=len(train),forecast_origin_start=str(dates[origins[batch[0]]].date()),forecast_origin_end=str(dates[origins[batch[-1]]].date()),candidate_columns=len(features),model_input_columns=len(features)+len(ids),estimator="Random Forest",trees=params["n_estimators"]))
        print('Completed RF fit:',dates[fit].date(),flush=True)
    assert np.isfinite(pred).all()
    np.savez_compressed(output/'predictions.npz',raw_predictions=pred,actual_cases=actual,zip=ids,forecast_origins=dates[origins].to_numpy(),horizons=[1,2,3])
    pd.DataFrame(logs).to_csv(output/'training_timing.csv',index=False)
    pd.DataFrame(importance).to_csv(output/'training_impurity_importance.csv',index=False)
    (output/'model_parameters.json').write_text(json.dumps(dict(parameters=params,sklearn_version=sklearn.__version__,python_version=platform.python_version(),tuning='None; fixed before viewing this run',region_identity='585 one-hot ZIP identity columns, same region identity available to FE benchmark. Not numeric postal ordering.',static_attributes='Not added; matched34-column candidate information set'),indent=2),encoding='utf8')
    pd.DataFrame([(g,f) for g,fs in GROUPS.items() for f in fs],columns=['group','feature']).to_csv(output/'candidate_features.csv',index=False)
    p=np.maximum(pred,0);err=p-actual;da=np.diff(actual,axis=2);dp=np.diff(p,axis=2)
    aa=np.diff(actual,n=2,axis=2)[...,0];ap=np.diff(p,n=2,axis=2)[...,0]
    daily=pd.DataFrame(dict(forecast_origin=dates[origins],trajectory_MAE=np.abs(err).mean(axis=(0,2)),MSE=(err**2).mean(axis=(0,2)),mean_bias=err.mean(axis=(0,2)),slope_MAE=np.abs((p[...,2]-p[...,0]-actual[...,2]+actual[...,0])/2).mean(axis=0),direction_agreement=(np.sign(p[...,2]-p[...,0])==np.sign(actual[...,2]-actual[...,0])).mean(axis=0),two_step_shape_agreement=(np.sign(dp)==np.sign(da)).all(axis=2).mean(axis=0),change_MAE=np.abs(dp-da).mean(axis=(0,2)),acceleration_MAE=np.abs(ap-aa).mean(axis=0),all_zero_actual_path_share=(actual==0).all(axis=2).mean(axis=0)))
    daily.to_csv(output/'scores_by_forecast_origin.csv',index=False)
    rows=[]
    periods={'overall':np.repeat('all',len(daily)),'year':daily.forecast_origin.dt.year.astype(str).to_numpy(),'quarter':daily.forecast_origin.dt.to_period('Q').astype(str).to_numpy()}
    for scope,values in periods.items():
        for label in np.unique(values):
            d=daily.loc[values==label];r=dict(scope=scope,period=label,forecast_origins=len(d),regions=len(ids),paths=len(d)*len(ids),RMSE=float(np.sqrt(d.MSE.mean())))
            r.update({col:float(d[col].mean()) for col in daily.columns if col not in ['forecast_origin','MSE']});rows.append(r)
    scores=pd.DataFrame(rows);scores.to_csv(output/'trajectory_scores_by_period.csv',index=False)
    protocol=dict(stage=stage,initial_training=['2020-03-01','2020-04-29'],development=['2020-04-30','2020-12-31'],evaluation=['2021-01-01','2023-12-31'],window_calendar_days=60,retrain_every_days=7,first_fit='2020-04-29 end of day',prediction='Daily end-of-day origin predicts t+1,t+2,t+3; raw onset counts',boundaries='All3 targets remain within stage; development origins end Dec28 2020, evaluation origins end Dec28 2023. December29-31 origins omitted to avoid crossing targets.',maturity='Every training target <=fit date; forecast targets >forecast origin>=fit date',feature_policy='Same34 candidates including bothPressure plus one-hot ZIP identity. No new static attributes, no feature selection. Training-only tree splits; no early stopping on validation.',warmup='14 days inside each60-day window,45/44/43 usable training origins per ZIP per horizon; sparse-history state reset at window start',inputs='Updated daily through origin; coefficients updated only on anchored7-day schedule',evaluation_policy='Algorithm and settings frozen before subsequent-period run. Past realized evaluation labels may enter future scheduled fits; no retrospective prediction changes or manual tuning.',limitation='Exploratory retrospective time evaluation;2021-2023 previously inspected. Onset-date availability assumed, reporting delays not modeled.',source_sha256=hashlib.sha256(source.read_bytes()).hexdigest())
    (output/'protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf8')
    (output/'verification.json').write_text(json.dumps(dict(all_training_targets_mature=True,window_exactly60=True,forecast_targets_after_fit=True,all_predictions_finite=True,all34_candidates_considered=True,per_zip_metrics_produced=False,no_validation_early_stopping=True),indent=2))
    (output/'RESULTS.md').write_text('# Random Forest rolling60-day trajectory evaluation\n\n'+scores.to_string(index=False)+'\n\n60 calendar days,14 within-window warmup days;45/44/43 supervised origins per horizon per ZIP. Weekly re-estimation, daily predictor updates. No Pressure ablation or seasonal naive. Primary score is ZIP-path trajectory MAE; change/acceleration and direction/shape are supplementary. Scores are retrospective and exploratory. See protocol.json and training_timing.csv for information boundaries. No coefficient p-values or causal feature interpretations are asserted for this Random Forest estimator.\n',encoding='utf8')
    return scores, output


In [ ]:
ROOT=Path.cwd()
if not (ROOT/'rolling_trajectory_outputs').exists() and (ROOT/'Weikun').is_dir():
    ROOT=ROOT/'Weikun'
SOURCE=ROOT/'rolling_trajectory_outputs'/'inputs'/'nj_daily_features.csv.gz'
COHORT=ROOT/'fixed60_trajectory_outputs'/'fixed60_20261027'/'common_regions.csv'
DP_RUN=ROOT/'rolling_trajectory_outputs'/'rolling60_weekly_development_20261027'
XGB_RUN=ROOT/'xgboost_trajectory_outputs'/'xgb60_weekly_development_20261027'
OUTPUT=ROOT/'random_forest_trajectory_outputs'/'rf60_weekly_development_20261027'
# This refits105 forests; use final saved-results cell if you only want to inspect.
rf_scores,folder=run_random_forest60(SOURCE,COHORT,OUTPUT,stage='development')
display(rf_scores)
print(folder.resolve())

## Matched comparisons and sensitivity
All archives must match IDs/order,actual counts,dates andhorizons. Primary:raw-count trajectoryMAE. Direction and shape use exact-zero classification as before. Supporting outcomes includeRMSE,bias,velocity/acceleration magnitude errors,class recalls,calendarquarters/months and selected actual-pathgroups.

Three planned pairwiseMAE contrasts receive exploratory date-levelHAC tests (lag7),Holm adjusted across allthree. Seven-day moving date-block CIs are primary uncertainty descriptions,14-day blocks sensitivity. Regions stay together in each draw. These are not prediction intervals or coefficient p-values; nonstationary epidemics and prior exploratory decisions limit inference.

Deadbands0,0.5,1cases/day were already introduced inNotebook21. Zero remains primary. Applying a deadband changes actual labels and predicted labels as well as class balance. It is not post-hoc tuning, and we do not pick the threshold with the highest accuracy or use it to declare a winner.

In [ ]:
from pathlib import Path
from itertools import combinations
import json,hashlib
import numpy as np
import pandas as pd

def three_model_analysis(archives,output):
    """Matched retrospective diagnostics, not hyperparameter selection."""
    from scipy.stats import norm
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    loaded={};manifest={};ref=None
    for name,path in archives.items():
        with np.load(path,allow_pickle=False) as z:
            if ref is None:ref={k:z[k].copy() for k in ['zip','forecast_origins','horizons','actual_cases']}
            for k,v in ref.items():assert np.array_equal(z[k],v),f'{name}: {k} differs'
            loaded[name]=np.maximum(z['raw_predictions'].copy(),0)
        manifest[name]=dict(path=str(path),sha256=hashlib.sha256(Path(path).read_bytes()).hexdigest())
    a=ref['actual_cases'];dates=pd.to_datetime(ref['forecast_origins']);h=ref['horizons'];N,T,H=a.shape
    assert (N,T,H)==(585,243,3) and np.array_equal(h,[1,2,3])
    day={};summary=[];quarter=[];monthly=[];groups=[];horizons=[];classes=[];sensitivity=[]
    da=np.diff(a,axis=2);aa=np.diff(a,n=2,axis=2)[...,0];ad=np.sign(a[...,2]-a[...,0])
    for name,p in loaded.items():
        e=p-a;dp=np.diff(p,axis=2);ap=np.diff(p,n=2,axis=2)[...,0];pdirection=np.sign(p[...,2]-p[...,0])
        cm=np.array([[((ad==x)&(pdirection==y)).sum() for y in [-1,0,1]] for x in [-1,0,1]])
        for i,label in enumerate(['Declining','Endpoint-flat','Rising']):
            classes.append(dict(model=name,actual_class=label,actual_paths=int(cm[i].sum()),recall=cm[i,i]/cm[i].sum(),precision=cm[i,i]/cm[:,i].sum() if cm[:,i].sum() else np.nan))
        d=pd.DataFrame(dict(forecast_origin=dates,trajectory_MAE=np.abs(e).mean(axis=(0,2)),MSE=(e**2).mean(axis=(0,2)),mean_bias=e.mean(axis=(0,2)),direction_agreement=(ad==pdirection).mean(axis=0),shape_agreement=(np.sign(da)==np.sign(dp)).all(axis=2).mean(axis=0),change_MAE=np.abs(dp-da).mean(axis=(0,2)),acceleration_MAE=np.abs(ap-aa).mean(axis=0)))
        day[name]=d
        row=dict(model=name,regions=N,origins=T,paths=N*T,RMSE=float(np.sqrt(d.MSE.mean())),balanced_direction_accuracy=float(np.mean(np.diag(cm)/cm.sum(axis=1))),actual_endpoint_flat_share=float((ad==0).mean()),actual_all_zero_share=float((a==0).all(axis=2).mean()))
        row.update({k:float(d[k].mean()) for k in d if k not in ['forecast_origin','MSE']});summary.append(row)
        for freq,dest in [('Q',quarter),('M',monthly)]:
            for label,f in d.groupby(d.forecast_origin.dt.to_period(freq)):
                r=dict(model=name,period=str(label),origins=len(f),RMSE=float(np.sqrt(f.MSE.mean())))
                r.update({k:float(f[k].mean()) for k in f if k not in ['forecast_origin','MSE']});dest.append(r)
        for label,mask in [('all_paths',np.ones((N,T),bool)),('all_three_zero',(a==0).all(axis=2)),('endpoint_nonflat',ad!=0),('endpoint_flat',ad==0)]:
            groups.append(dict(model=name,group=label,paths=int(mask.sum()),MAE=float(np.abs(e[mask]).mean()),direction_agreement=float((ad[mask]==pdirection[mask]).mean()),shape_agreement=float((np.sign(da[mask])==np.sign(dp[mask])).all(axis=1).mean())))
        for hh,horizon in enumerate(h):
            er=e[...,hh];horizons.append(dict(model=name,horizon=int(horizon),MAE=float(np.abs(er).mean()),RMSE=float(np.sqrt((er**2).mean())),bias=float(er.mean())))
        # Prespecified sensitivity only, never choose the best-looking threshold.
        for threshold in [0.,0.5,1.]:
            classify=lambda x:np.where(x>threshold,1,np.where(x<-threshold,-1,0))
            actual_sign=classify((a[...,2]-a[...,0])/2);pred_sign=classify((p[...,2]-p[...,0])/2)
            ac=classify(da);pc=classify(dp)
            recalls=[(pred_sign[actual_sign==j]==j).mean() if (actual_sign==j).any() else np.nan for j in [-1,0,1]]
            sensitivity.append(dict(model=name,threshold=threshold,slope_threshold_units='cases/day',movement_threshold_units='cases/day with1-day spacing',direction_agreement=float((actual_sign==pred_sign).mean()),balanced_direction_accuracy=float(np.nanmean(recalls)),actual_endpoint_flat_share=float((actual_sign==0).mean()),always_flat_direction_reference=float((actual_sign==0).mean()),two_step_shape_agreement=float((ac==pc).all(axis=2).mean()),actual_both_steps_flat_share=float((ac==0).all(axis=2).mean()),primary=threshold==0))
    summary=pd.DataFrame(summary);summary.to_csv(output/'three_model_overall.csv',index=False)
    for filename,data in [('quarter_comparison.csv',quarter),('month_comparison.csv',monthly),('path_group_comparison.csv',groups),('horizon_comparison.csv',horizons),('direction_class_recall.csv',classes),('direction_threshold_sensitivity.csv',sensitivity)]:pd.DataFrame(data).to_csv(output/filename,index=False)
    # Common date resampling preserves covariance across all models and regions.
    rng=np.random.default_rng(20261027);draws={}
    for block in [7,14]:
        start=rng.integers(0,T-block+1,(2000,int(np.ceil(T/block))))
        draws[block]=(start[...,None]+np.arange(block)).reshape(2000,-1)[:,:T]
    contrasts=[];primary_tests=[];daily_contrasts=[]
    for base,candidate in combinations(loaded.keys(),2):
        for metric in ['trajectory_MAE','direction_agreement','shape_agreement','change_MAE','acceleration_MAE']:
            diff=(day[candidate][metric]-day[base][metric]).to_numpy() if 'agreement' in metric else (day[base][metric]-day[candidate][metric]).to_numpy()
            for block,ix in draws.items():
                lo,hi=np.quantile(diff[ix].mean(axis=1),[.025,.975])
                contrasts.append(dict(reference=base,candidate=candidate,metric=metric,improvement=float(diff.mean()),ci95_low=float(lo),ci95_high=float(hi),block_days=block,definition='Positive favors candidate. Pointwise exploratory date-block interval, conditional on fitted models and this historical period.'))
            if metric=='trajectory_MAE':
                # HAC test on one regional-average paired loss series, not142k IID paths.
                centered=diff-diff.mean();lag=7;lrv=float(centered@centered/T)
                for k in range(1,lag+1):lrv+=2*(1-k/(lag+1))*float(centered[k:]@centered[:-k]/T)
                se=float(np.sqrt(max(lrv,0)/T));z=float(diff.mean()/se) if se>1e-15 else np.nan
                primary_tests.append(dict(reference=base,candidate=candidate,MAE_improvement=float(diff.mean()),HAC_SE=se,HAC_z=z,p_HAC_two_sided=float(2*norm.sf(abs(z))) if np.isfinite(z) else np.nan,HAC_lags=lag,dates=T,interpretation='Exploratory asymptotic paired loss test on meanZIP loss per date; nonstationarity/dependence and prior exploration limit inference. Not a model coefficient or causal test.'))
                daily_contrasts.extend(dict(reference=base,candidate=candidate,forecast_origin=str(date.date()),MAE_improvement=float(value)) for date,value in zip(dates,diff))
    tests=pd.DataFrame(primary_tests);valid=np.flatnonzero(np.isfinite(tests.p_HAC_two_sided));order=valid[np.argsort(tests.p_HAC_two_sided.to_numpy()[valid])];adjust=np.maximum.accumulate(tests.p_HAC_two_sided.to_numpy()[order]*(len(order)-np.arange(len(order))))
    tests['p_Holm']=np.nan;tests.loc[order,'p_Holm']=np.minimum(adjust,1)
    tests.to_csv(output/'primary_MAE_HAC_Holm_tests.csv',index=False)
    pd.DataFrame(contrasts).to_csv(output/'paired_date_block_intervals.csv',index=False)
    pd.DataFrame(daily_contrasts).to_csv(output/'paired_daily_MAE_improvements.csv',index=False)
    (output/'analysis_definition.json').write_text(json.dumps(dict(models=manifest,primary='Raw-count trajectory MAE, threshold0 direction is supporting; never select a sign threshold by its validation accuracy',thresholds=[0,0.5,1],threshold_interpretation='Different deadbands change truth labels, predicted labels and class balance; higher agreement does not imply improved original target prediction.',p_family='Three pairwise MAE differences, Holm correction. Supporting metrics have pointwise CIs only, no unadjusted p claims.',ci='Common moving date-block draws7days primary/14days sensitivity,2000; no IID region assumption',report_role='Exploratory2020 rolling validation with previously developed features/cohort; not final holdout.',no_new_features=True,no_pressure_ablation=True,no_model_refit_in_comparison=True),indent=2),encoding='utf8')
    report='# Three-model trajectory comparison:2020 rolling validation\n\n'+summary.to_string(index=False)+'\n\n## Reading the analysis\nCompare count MAE, RMSE, exact direction/shape and magnitude-of-change errors separately. Stratify by quarter and actual path group; groups overlap and do not partition all paths. Geographic diagnostics are separate. Parameters were fixed before the RF run, not optimized to validation scores.\n\n## Uncertainty\nPaired differences average losses across all ZIPs at each date before inference. Seven-day and14-day moving-block intervals are pointwise and exploratory. The primary MAE family includes three pairwise HAC tests, Holm adjusted. This is not a coefficient significance test, a clinical efficacy claim or a calibrated prediction interval. Nonstationary outbreaks and overlapping forecasts limit both asymptotic p-values and block intervals.\n\n## Strict-zero sensitivity\nThe same raw predictions are evaluated under0,0.5,1 cases/day deadbands, introduced as sensitivity values in Notebook21. Threshold0 remains primary. Threshold changes also alter actual labels and the always-flat reference; do not pick the threshold maximizing a model score. Exact-flat scores can be low for small continuous predictions even when count errors are small.\n'
    (output/'RESULTS.md').write_text(report,encoding='utf8')
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    names=list(loaded);short=['Dynamic Panel','XGBoost','Random Forest'];colors=['#64748b','#2563eb','#087f78']
    fig,axes=plt.subplots(1,3,figsize=(14,4))
    for ax,col,title in zip(axes,['trajectory_MAE','direction_agreement','shape_agreement'],['Trajectory MAE (cases)','Strict endpoint direction agreement','Strict two-step shape agreement']):
        bars=ax.bar(short,summary[col],color=colors);ax.set(title=title,ylim=(0,float(summary[col].max())*1.25));ax.grid(axis='y',alpha=.2)
        ax.bar_label(bars,labels=[f'{v:.1%}' if 'agreement' in col else f'{v:.3f}' for v in summary[col]],padding=3)
    fig.suptitle('Matched 2020 rolling validation | 585 ZIPs | 243 daily origins | 3-day paths')
    fig.tight_layout();fig.savefig(output/'01_three_model_comparison.png',dpi=240);plt.close(fig)
    fig,axes=plt.subplots(1,3,figsize=(16,4))
    for ax,col,title in zip(axes,['direction_agreement','two_step_shape_agreement','balanced_direction_accuracy'],['Endpoint direction agreement','Two-step shape agreement','Balanced direction accuracy']):
        for name,label,color in zip(names,short,colors):
            d=pd.DataFrame(sensitivity);d=d.loc[d.model.eq(name)];ax.plot(d.threshold,d[col],marker='o',label=label,color=color)
        ref=pd.DataFrame(sensitivity).loc[lambda d:d.model.eq(names[0])]
        reference_values=ref.always_flat_direction_reference if col=='direction_agreement' else ref.actual_both_steps_flat_share if col=='two_step_shape_agreement' else np.repeat(1/3,3)
        ax.plot(ref.threshold,reference_values,color='#d94841',ls='--',label='Always-flat classification reference')
        ax.set(title=title,xlabel='Deadband threshold (cases/day)',ylabel='Agreement',xticks=[0,.5,1],ylim=(0,1));ax.grid(alpha=.2);ax.legend(frameon=False,fontsize=7)
    fig.suptitle('Sensitivity only: thresholds change labels; primary threshold remains zero')
    fig.tight_layout();fig.savefig(output/'02_direction_threshold_sensitivity.png',dpi=240);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    q=pd.DataFrame(quarter);group=pd.DataFrame(groups)
    x=np.arange(3)
    for i,(name,label,color) in enumerate(zip(names,short,colors)):
        d=q.loc[q.model.eq(name)];axes[0].bar(x+(i-1)*.24,d.trajectory_MAE,width=.24,label=label,color=color)
        d=group.loc[group.model.eq(name)].set_index('group');axes[1].bar(np.arange(2)+(i-1)*.24,d.loc[['all_three_zero','endpoint_nonflat'],'MAE'],width=.24,label=label,color=color)
    axes[0].set(title='Quarter-specific trajectory MAE',xticks=x,xticklabels=['Q2: Apr30 onward','Q3','Q4'],ylabel='MAE (cases)')
    axes[1].set(title='Selected actual path groups',xticks=[0,1],xticklabels=['All three days zero','Different endpoints'],ylabel='MAE (cases)')
    for ax in axes:ax.legend(frameon=False,fontsize=8);ax.grid(axis='y',alpha=.2)
    fig.tight_layout();fig.savefig(output/'03_quarter_and_path_groups.png',dpi=240);plt.close(fig)
    fig,axes=plt.subplots(4,1,figsize=(13,12),sharex=True)
    for name,label,color in zip(names,short,colors):axes[0].plot(dates,day[name].trajectory_MAE,label=label,color=color,lw=1)
    axes[0].set(title='ZIP-level trajectory MAE',ylabel='MAE (cases)');axes[0].legend(frameon=False,ncol=3)
    for hh,horizon in enumerate(h):
        ax=axes[hh+1];ax.plot(dates,a[...,hh].sum(axis=0),label='Actual',color='#111827',lw=1.4)
        for name,label,color in zip(names,short,colors):ax.plot(dates,loaded[name][...,hh].sum(axis=0),label=label,color=color,lw=1)
        ax.set(title=f'{int(horizon)}-day ahead: summed across 585 ZIPs',ylabel='Total daily cases');ax.legend(frameon=False,ncol=4,fontsize=8)
    for ax in axes:ax.grid(alpha=.2)
    axes[-1].set_xlabel('Forecast origin (target date=origin+horizon)')
    fig.suptitle('Actual and predicted activity alongside trajectory errors |2020 rolling validation')
    fig.text(.5,.012,'Aggregate curves can hide offsetting ZIP errors. No smoothing. Top panel scores ZIP-level paths, not aggregate totals.',ha='center',fontsize=9)
    fig.tight_layout(rect=[0,.035,1,.97]);fig.savefig(output/'04_errors_and_actual_forecasts.png',dpi=220);plt.close(fig)
    fig,axes=plt.subplots(1,4,figsize=(14,4))
    for ax,date in zip(axes,pd.to_datetime(['2020-04-30','2020-07-01','2020-10-01','2020-12-01'])):
        t=dates.get_loc(date);ax.plot(h,a[:,t,:].sum(axis=0),color='#111827',marker='o',label='Actual')
        for name,label,color in zip(names,short,colors):ax.plot(h,loaded[name][:,t,:].sum(axis=0),color=color,marker='o',label=label)
        ax.set(title=f'Origin: {date.date()}',xlabel='Days ahead',xticks=h);ax.grid(alpha=.2);ax.legend(frameon=False,fontsize=7)
    axes[0].set_ylabel('Total daily cases across 585 ZIPs')
    fig.suptitle('Same-origin three-day paths: four calendar-selected illustrative origins, not representative scores')
    fig.tight_layout();fig.savefig(output/'05_same_origin_path_examples.png',dpi=240);plt.close(fig)
    return summary,tests


In [ ]:
ARCHIVES={'Dynamic Panel FE':DP_RUN/'predictions.npz','XGBoost':XGB_RUN/'predictions.npz','Random Forest':OUTPUT/'predictions.npz'}
comparison,primary_tests=three_model_analysis(ARCHIVES,OUTPUT/'comparison')
display(comparison)
display(primary_tests)
display(pd.read_csv(OUTPUT/'comparison'/'quarter_comparison.csv'))
display(pd.read_csv(OUTPUT/'comparison'/'direction_threshold_sensitivity.csv'))

## Residual Moran's I
Same585 geographies,8-neighbor symmetric graph,row-normalization,999permutations/seed and same date/horizon family asNotebooks24/25. K4/K12 descriptive sensitivity. Exact ZIP–ZCTA identifier mapping remains unvalidated. Permutation inference is exploratory under residual exchangeability; unequal count-residual variances and spatial dependence can affect it. Different per-modelBHqvalues are not a paired significance test between models.

In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd

def residual_moran(archive, geography, output, permutations=999, seed=20261027):
    """Global residual Moran's I at each origin/horizon, no fitting."""
    archive,geography,output=Path(archive),Path(geography),Path(output)
    output.mkdir(parents=True,exist_ok=True)
    with np.load(archive,allow_pickle=False) as z:
        ids=z['zip'].astype(str);dates=pd.to_datetime(z['forecast_origins']);h=z['horizons']
        residual=np.maximum(z['raw_predictions'],0)-z['actual_cases']
    assert np.array_equal(h,[1,2,3]) and np.isfinite(residual).all()
    g=pd.read_csv(geography,sep='\t',dtype={'GEOID':str});g.columns=g.columns.str.strip()
    g['GEOID']=g.GEOID.str.zfill(5)
    assert not g.GEOID.duplicated().any()
    matched=g.set_index('GEOID').reindex(ids)
    coords=matched[['INTPTLAT','INTPTLONG']].apply(pd.to_numeric,errors='coerce').to_numpy()
    if not np.isfinite(coords).all():raise ValueError('Missing coordinates: do not silently drop regions')
    n,T,H=residual.shape;assert n==len(ids) and len(set(ids))==n
    # Haversine distance, internal representative points, not polygon centroids.
    lat,lon=np.deg2rad(coords).T
    dlat=lat[:,None]-lat;dlon=lon[:,None]-lon
    term=np.sin(dlat/2)**2+np.cos(lat[:,None])*np.cos(lat)*np.sin(dlon/2)**2
    distance=6371.0088*2*np.arcsin(np.sqrt(np.clip(term,0,1)))
    np.fill_diagonal(distance,np.inf)
    order=np.argsort(distance,axis=1,kind='stable') # stable tie-break = saved cohort order
    centered=residual.reshape(n,-1)-residual.reshape(n,-1).mean(axis=0)
    den=(centered**2).sum(axis=0);valid=den>1e-20;expected=-1/(n-1)
    def weights(k):
        binary=np.zeros((n,n),bool);binary[np.arange(n)[:,None],order[:,:k]]=True
        binary|=binary.T;np.fill_diagonal(binary,False)
        w=binary/binary.sum(axis=1)[:,None]
        assert np.allclose(w.sum(axis=1),1) and np.all(np.diag(w)==0)
        return w,binary
    def statistic(z,w):
        # Edge form avoids a new dependency on spatial-statistics packages.
        i,j=np.nonzero(w)
        return (n/w.sum())*np.einsum('e,ec,ec->c',w[i,j],z[i],z[j],optimize=False)/np.where(valid,den,np.nan)
    results=[];sensitivity=[]
    for k in [4,8,12]:
        w,binary=weights(k);observed=statistic(centered,w)
        sensitivity.append(observed)
        if k!=8:continue
        ei,ej=np.nonzero(w);ew=w[ei,ej]
        pd.DataFrame(dict(origin_zip=ids[ei],neighbor_zip=ids[ej],weight=ew,distance_km=distance[ei,ej])).to_csv(output/'spatial_weights_knn8.csv',index=False)
        pd.DataFrame(dict(zip=ids,latitude=coords[:,0],longitude=coords[:,1],neighbor_degree=binary.sum(axis=1),geography_status='exact_identifier_match_unvalidated')).to_csv(output/'geography_alignment.csv',index=False)
        rng=np.random.default_rng(seed);extreme=np.zeros(centered.shape[1],dtype=int)
        # One permutation of region labels for the whole residual cube preserves
        # each region's temporal/horizon profile. Tests remain marginal by date/h.
        for b in range(permutations):
            perm=rng.permutation(n)
            sim=statistic(centered[perm],w)
            extreme+=np.abs(sim-expected)>=np.abs(observed-expected)-1e-14
            if (b+1)%100==0:print(f'Moran permutations: {b+1}/{permutations}',flush=True)
        p=(extreme+1)/(permutations+1);p[~valid]=np.nan
        # BH across all valid origin-horizon tests in THIS stage/run.
        q=np.full_like(p,np.nan);idx=np.flatnonzero(valid);ranked=idx[np.argsort(p[idx])]
        q[ranked]=np.minimum(1,np.minimum.accumulate((p[ranked]*len(idx)/np.arange(1,len(idx)+1))[::-1])[::-1])
        for t,date in enumerate(dates):
            for hh,horizon in enumerate(h):
                c=t*H+hh
                results.append(dict(forecast_origin=str(date.date()),target_date=str((date+pd.Timedelta(days=int(horizon))).date()),horizon=int(horizon),regions=n,Moran_I=observed[c],expected_I=expected,p_permutation_two_sided=p[c],q_BH=q[c],residual_variance=float(den[c]/n),status='ok' if valid[c] else 'undefined_constant_residuals'))
        # Independent dense-form formula check on first valid residual vector.
        c=int(idx[0]);vec=centered[:,c]
        assert np.isclose(observed[c],n/w.sum()*(vec@w@vec)/(vec@vec))
        assert np.allclose(statistic(centered+0,w),statistic(-centered,w),equal_nan=True)
    table=pd.DataFrame(results);table.to_csv(output/'residual_moran_by_origin_horizon.csv',index=False)
    sens=[]
    for k,values in zip([4,8,12],sensitivity):
        for hh,horizon in enumerate(h):
            v=values.reshape(T,H)[:,hh];sens.append(dict(k=k,horizon=int(horizon),mean_Moran_I=float(np.nanmean(v)),median_Moran_I=float(np.nanmedian(v)),positive_I_share=float(np.mean(v>0)),inference='Descriptive sensitivity only; no p-values for alternative k'))
    pd.DataFrame(sens).to_csv(output/'weight_sensitivity.csv',index=False)
    summary=[]
    for scope,labels in [('overall',np.repeat('all',len(table))),('quarter',pd.to_datetime(table.forecast_origin).dt.to_period('Q').astype(str).to_numpy()),('year',pd.to_datetime(table.forecast_origin).dt.year.astype(str).to_numpy())]:
        for label in np.unique(labels):
            for horizon in h:
                d=table.loc[(labels==label)&table.horizon.eq(horizon)&table.status.eq('ok')]
                summary.append(dict(scope=scope,period=label,horizon=int(horizon),valid_tests=len(d),mean_Moran_I=d.Moran_I.mean(),median_Moran_I=d.Moran_I.median(),raw_p_below_05_share=d.p_permutation_two_sided.lt(.05).mean(),BH_q_below_05_share=d.q_BH.lt(.05).mean(),positive_I_and_BH_q_below_05_share=(d.Moran_I.gt(0)&d.q_BH.lt(.05)).mean()))
    summary=pd.DataFrame(summary);summary.to_csv(output/'residual_moran_summary.csv',index=False)
    protocol=dict(metric="Global residual Moran's I",residual='max(raw_prediction,0)-actual; sign convention does not change I',alignment='At each forecast origin and horizon, compare585 ZIP residuals; never flatten region-days or sum across regions',spatial_weights='Symmetric union of8 nearest neighbors using haversine distances between2021 Gazetteer internal points; zero diagonal, row-standardized after symmetrizing. Weights after normalization are not necessarily symmetric.',primary_k=8,sensitivity_k=[4,12],distance='Great-circle km; internal points, not polygon centroid or adjacency',mapping='Exact postal ZIP identifier to ZCTA GEOID remains unvalidated; diagnostic conditional on this mapping',null='Exchangeability of residual labels across regions conditional on fitted predictions; heteroskedastic count residuals may violate it. Exploratory diagnostic, not a formal spatial-regression specification test.',permutations=permutations,seed=seed,p_value='(1+count(|I_perm-EI|>=|I_obs-EI|))/(B+1), two-sided centered atEI=-1/(n-1)',multiple_testing='Benjamini-Hochberg across all valid dates x horizons in this stage; overlapping dates/spatial dependence mean FDR control is not guaranteed. No pooled overall significance claim.',constant_residual_policy='Undefined I and p, retained as NA',parameter_fitting='None. No prediction or feature changes.',inference='Positive I indicates similar signed residuals cluster under selected W. Nonsignificance is not evidence that all spatial patterns are explained.',reference='https://pysal.org/esda/stable/user-guide/global_morans_i.html',archive_sha256=hashlib.sha256(archive.read_bytes()).hexdigest(),geography_sha256=hashlib.sha256(geography.read_bytes()).hexdigest())
    (output/'residual_moran_definition.json').write_text(json.dumps(protocol,indent=2),encoding='utf8')
    (output/'verification.json').write_text(json.dumps(dict(all585_coordinates_present=True,weights_zero_diagonal=True,weights_rows_sum_to_one=True,dense_formula_crosscheck=True,sign_invariance_verified=True,predictions_unchanged=True,tests=len(table)),indent=2))
    report='# Residual spatial autocorrelation: rolling60 weekly model\n\n'+summary.loc[summary.scope.eq('overall')].to_string(index=False)+'\n\nMoran I is computed separately for each origin and horizon. A positive value suggests clustering of similarly signed forecast errors under the chosen geographic weights. Summary means are descriptive; they do not have a pooled p-value. Permutation p-values and BH q-values are exploratory, conditional on the fitted model, residual exchangeability, and unvalidated ZIP-ZCTA identifiers. Count residual magnitudes vary by region; this can affect the interpretation. Nonsignificance does not prove spatial adequacy. K4/K12 are sensitivity checks, not selected by results. No model refit.\n'
    (output/'RESULTS.md').write_text(report,encoding='utf8')
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        fig,axes=plt.subplots(3,1,figsize=(12,8),sharex=True)
        for ax,horizon in zip(axes,h):
            d=table.loc[table.horizon.eq(horizon)];x=pd.to_datetime(d.forecast_origin)
            ax.plot(x,d.Moran_I,color='#2563eb',lw=1,label="Residual Moran's I")
            sig=d.q_BH.lt(.05);ax.scatter(x[sig],d.loc[sig,'Moran_I'],s=9,color='#d94841',label='BH q <0.05 (exploratory)')
            ax.axhline(expected,color='#64748b',ls='--',lw=1,label='Randomization expectation')
            ax.set(ylabel="Moran's I",title=f'{horizon}-day ahead');ax.grid(alpha=.2)
        axes[0].legend(frameon=False,ncol=3,fontsize=8);axes[-1].set_xlabel('Forecast origin')
        fig.suptitle('Residual spatial clustering |8-neighbor geographic weights |2020 rolling validation')
        fig.tight_layout();fig.savefig(output/'residual_moran_by_origin.png',dpi=220);plt.close(fig)
    except ImportError:print('Optional plot requires matplotlib')
    return table,summary


In [ ]:
moran_daily,moran_summary=residual_moran(OUTPUT/'predictions.npz',ROOT/'rolling_trajectory_outputs'/'inputs'/'2021_Gaz_zcta_national.txt',OUTPUT/'residual_moran')
spatial=[]
for name,run in [('Dynamic Panel FE',DP_RUN),('XGBoost',XGB_RUN),('Random Forest',OUTPUT)]:
    d=pd.read_csv(run/'residual_moran'/'residual_moran_summary.csv')
    d=d.loc[d.scope.eq('overall')].copy();d.insert(0,'model',name);spatial.append(d)
spatial_comparison=pd.concat(spatial)
spatial_comparison.to_csv(OUTPUT/'comparison'/'residual_moran_comparison.csv',index=False)
display(spatial_comparison)

## Read completed outputs without retraining
After reopening, run only this cell to inspect all saved results and figures. Model pickle/joblib files should only be loaded from your own trusted experiment folder, and require compatiblelibrary versions. Impurity importance is a training diagnostic, not causal importance or statistical significance.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image,display
SAVED_ROOT=Path.cwd()
if not (SAVED_ROOT/'random_forest_trajectory_outputs').exists() and (SAVED_ROOT/'Weikun').is_dir():
    SAVED_ROOT=SAVED_ROOT/'Weikun'
SAVED=SAVED_ROOT/'random_forest_trajectory_outputs'/'rf60_weekly_development_20261027'/'comparison'
for filename in ['three_model_overall.csv','primary_MAE_HAC_Holm_tests.csv','quarter_comparison.csv','path_group_comparison.csv','direction_threshold_sensitivity.csv','residual_moran_comparison.csv']:
    print(filename);display(pd.read_csv(SAVED/filename))
for filename in ['01_three_model_comparison.png','02_direction_threshold_sensitivity.png','03_quarter_and_path_groups.png','04_errors_and_actual_forecasts.png','05_same_origin_path_examples.png']:
    display(Image(filename=str(SAVED/filename)))
print((SAVED/'RESULTS.md').read_text(encoding='utf8'))